## Executive summary

A sensible water tank can serve two very different roles:  
(a) an **inertial storage / buffer tank** used to stabilize flow rates, reduce ON/OFF cycling, and support defrosting;  
(b) a **thermal store / TES** used for intentional storage, such as load shifting, peak shaving, PV integration, or backup operation.  
Confusing these two roles almost always leads to over- or under-sizing.

Rules of thumb based on heat pump power are typically intended for case (a). The European standard **EN 15450** (design guideline for heat pump systems) reports a guideline value for buffer volume of **12–35 L per kW** of maximum heat pump capacity. In the literature, the German guideline **VDI 4645** is often referenced with intervals of **20–40 L/kW** depending on the application and assumptions. In the Italian context, one industrial manufacturer indicates **10–20 L/kW** as an indicative criterion and also proposes an **energy-based approach** based on minimum operating time and allowable temperature difference.

By contrast, when the objective is case (b), namely load shifting or peak shaving, the relevant quantity is the **useful stored energy (kWh)**, so the scaling changes quickly. The thermal capacity of water is about **1.16 kWh per m³·K** (equivalently, **0.00116 kWh per L·K**), consistent with the expressions reported in EN 15450 for the energy stored in a water volume. As a consequence, buffer tanks in the **100–300 L** range, while useful for anti-short-cycling purposes, often provide storage durations on the order of only a few minutes if used as TES under loads of several kW.

The choice of usable temperature lift **ΔT** is a crucial multiplier. In recent studies on optimal sizing and control, storage ΔT values range approximately from a few kelvin up to about **20 K**, depending on the use case and the control strategy. Increasing temperatures and ΔT improves TES energy density, but tends to worsen heat pump COP: in one example for district heating conditions, at an outdoor temperature of **5°C**, COP decreases from operating conditions **30/35°C** to **50/55°C**.

If TES is used in **MPC**, stratification (**see next section**) becomes a key issue. **One-node models** (perfectly mixed tanks) are often overly optimistic. Experimental work shows that a **three-node model**, capable of representing stratification, can significantly improve load-shifting performance compared to a one-node model, while reducing plant-model mismatch. **Control-oriented** multi-node tank models are therefore widely studied precisely because they keep complexity compatible with predictive control.

Finally, if the tank is involved in **DHW**, hygienic constraints must be considered. Italian guidance reports that hot water networks maintained above **50°C** are less frequently colonized, and that above **60°C** legionella activity is inhibited as a function of time. This leads in practice to accumulation temperatures around **55–60°C** with downstream mixing valves. **EN 15450** also requires that the system be able to reach **60°C daily, if required**.


## Stratification in TES

Thermal stratification in a sensible water tank refers to the formation of vertical layers at different temperatures, typically with hotter water in the upper region and colder water in the lower region. This phenomenon is important because it determines how much of the stored energy is actually usable during charging and discharging.

For modelling purposes, a perfectly mixed tank assumes a uniform temperature throughout the whole volume. This approximation is simple and computationally attractive, but it may overestimate the effective flexibility of the storage system. In contrast, stratified formulations preserve at least part of the vertical temperature gradient and therefore provide a more realistic representation of the available thermal energy.

In control-oriented applications, multi-node models are often used as a compromise between physical realism and computational tractability. Instead of resolving the full fluid dynamics inside the tank, the storage volume is divided into a small number of layers, each associated with its own temperature state. This allows the model to capture the main effects of stratification while remaining suitable for MPC.

<div style="text-align: center;">
  <img src="Alyphib_Figs/multinode.png" alt="Multi-node TES representation" width="420">
</div>

The figure below highlights the difference between idealized perfect stratification, experimentally observed stratification with a thermocline region, and the limiting case of complete mixing. For MPC-oriented modelling, the objective is usually not to reproduce every internal phenomenon in detail, but to retain enough structure to describe the evolution of useful energy within the tank.

<div style="text-align: center;">
  <img src="Alyphib_Figs/strastification.png" alt="Stratification concepts" width="560">
</div>


## Sizing logic: from building thermal demand to HP and TES models

The sizing-oriented code snippets presented in the following sections are intended to be **adaptive** rather than tied to a single fixed case study. The underlying idea is that the building-side thermal demand is not imposed here a priori: it will depend on the thermal characteristics of the building model, which are defined separately.

In practice, the building model will determine the required thermal load profile and, in particular, the relevant design or peak demand. Once this quantity is available, the heat pump and thermal storage models can be parameterized accordingly. In this sense, the HP and TES models proposed here should be interpreted as **scalable components**, whose main parameters are updated consistently with the thermal request coming from the building side.

This workflow is especially useful when detailed design data are not yet available. At an early stage, one may rely on simplified load estimates, for example using specific thermal demand values in W/m², while explicitly acknowledging that these are preliminary assumptions. At a later stage, the same modelling structure can be reused with more accurate building-side inputs, without changing the overall logic of the implementation.

Therefore, the purpose of the following code is not to define a single final plant size, but rather to provide a flexible framework in which:

- the building model supplies the thermal demand level,
- the heat pump model is scaled to match the required capacity,
- the TES model is sized consistently with the intended role of the storage, such as buffer operation or energy shifting.

Under this approach, once the building-side assumptions are refined, the HP and TES representations can be updated accordingly with minimal changes to the code structure.


## Heat pump modelling options compatible with an RC building model

If the building is represented through an equivalent RC thermal network, the cleanest way to couple the heat pump is to expose the HP as a model that delivers **thermal power** to the rest of the system. In other words, the building-side model does not need refrigerant states or detailed compressor physics: it only needs a consistent estimate of delivered heat and, when relevant, electrical consumption.

From a control-oriented perspective, this suggests a hierarchy of possible heat pump representations. The simplest version is an ideal thermal actuator. A more realistic but still lightweight version uses temperature-dependent capacity and COP maps. A future extension may include first-order dynamics to better represent actuator response and part-load transitions. The three approaches below should therefore be interpreted as modelling levels of increasing realism and complexity.


### Approach A — ideal thermal actuator

This is the simplest possible heat pump representation. The unit is treated as an ideal thermal actuator driven by a normalized control signal. The delivered thermal power is a fixed fraction of the nominal capacity, while electrical power is obtained from a constant COP.

This model is mainly useful as a **baseline or debugging tool**. It is easy to couple with a tank or building model, and it is often enough for first sanity checks. However, it does not capture the effects of source temperature, sink temperature, or part-load efficiency.


In [ ]:
from dataclasses import dataclass


@dataclass
class IdealHeatPumpParameters:
    """
    Parameters for the simplest heat pump representation.

    Attributes
    ----------
    q_nominal_kw : float
        Nominal thermal power delivered by the heat pump [kW].
    cop_constant : float
        Constant coefficient of performance [-].
    """

    q_nominal_kw: float
    cop_constant: float


class IdealHeatPump:
    """
    Simplest control-oriented heat pump model.

    The heat pump is represented as an ideal thermal actuator:
        Q_hp = u * Q_nominal
        P_hp = Q_hp / COP

    where:
    - u is a normalized control input in [0, 1]
    - Q_hp is the delivered thermal power [kW]
    - P_hp is the electrical power consumption [kW]
    """

    def __init__(self, params: IdealHeatPumpParameters) -> None:
        if params.q_nominal_kw <= 0:
            raise ValueError("q_nominal_kw must be strictly positive.")
        if params.cop_constant <= 0:
            raise ValueError("cop_constant must be strictly positive.")

        self.params = params

    def evaluate(self, control_signal: float) -> tuple[float, float]:
        """
        Evaluate heat pump thermal and electrical power.

        Parameters
        ----------
        control_signal : float
            Normalized control input in [0, 1].

        Returns
        -------
        q_hp_kw : float
            Delivered thermal power [kW].
        p_hp_kw : float
            Electrical power consumption [kW].
        """
        u = max(0.0, min(1.0, control_signal))

        q_hp_kw = u * self.params.q_nominal_kw
        p_hp_kw = q_hp_kw / self.params.cop_constant

        return q_hp_kw, p_hp_kw


# Minimal example
hp = IdealHeatPump(
    IdealHeatPumpParameters(
        q_nominal_kw=10.0,
        cop_constant=3.2
    )
)

u = 0.65
q_hp_kw, p_hp_kw = hp.evaluate(u)

print(f"Control signal: {u:.2f}")
print(f"Delivered thermal power: {q_hp_kw:.2f} kW")
print(f"Electrical power: {p_hp_kw:.2f} kW")


### Approach B — quasi-steady heat pump with capacity and COP maps

This is the recommended **entry-level model** for coupling with a building RC network and a simple TES representation. The heat pump remains algebraic and control-oriented, but both available capacity and COP depend on source-side and sink-side temperatures.

In practice, this means that the model should ideally be parameterized from **manufacturer performance maps**, technical datasheets, EN 14511 rating tables, or other fitted performance points. Even if only a few operating points are available, a simple affine or bilinear approximation is often sufficient for an initial implementation.

Compared with Approach A, this formulation already captures the main physical idea that warmer source temperatures improve performance, while higher sink temperatures tend to reduce both capacity and efficiency. At the same time, it remains easy to integrate into a tank-building workflow.


In [ ]:
from dataclasses import dataclass


@dataclass
class QuasiSteadyHeatPumpParameters:
    """
    Parameters for a simple quasi-steady heat pump model.

    Capacity map:
        Q_max = q_ref_kw
                + alpha_q_src * (T_source - t_source_ref_c)
                + alpha_q_sink * (T_sink - t_sink_ref_c)

    COP map:
        COP = cop_ref
              + alpha_cop_src * (T_source - t_source_ref_c)
              + alpha_cop_sink * (T_sink - t_sink_ref_c)

    Notes
    -----
    - alpha_q_src is usually positive: warmer source improves capacity.
    - alpha_q_sink is usually negative: hotter sink reduces capacity.
    - alpha_cop_src is usually positive: warmer source improves COP.
    - alpha_cop_sink is usually negative: hotter sink worsens COP.
    """

    q_ref_kw: float
    cop_ref: float
    t_source_ref_c: float
    t_sink_ref_c: float

    alpha_q_src: float
    alpha_q_sink: float
    alpha_cop_src: float
    alpha_cop_sink: float

    min_cop: float = 1.1
    min_modulation: float = 0.0
    max_modulation: float = 1.0


class QuasiSteadyHeatPump:
    """
    Simple control-oriented heat pump model based on source/sink temperature maps.

    Inputs
    ------
    - control_signal in [0, 1]
    - source temperature [°C]
    - sink temperature [°C]

    Outputs
    -------
    - delivered thermal power [kW]
    - electrical power [kW]
    - available maximum thermal power [kW]
    - COP [-]
    """

    def __init__(self, params: QuasiSteadyHeatPumpParameters) -> None:
        if params.q_ref_kw <= 0:
            raise ValueError("q_ref_kw must be strictly positive.")
        if params.cop_ref <= 0:
            raise ValueError("cop_ref must be strictly positive.")
        if params.min_cop <= 0:
            raise ValueError("min_cop must be strictly positive.")
        if not (0.0 <= params.min_modulation <= params.max_modulation <= 1.0):
            raise ValueError("Modulation bounds must satisfy 0 <= min <= max <= 1.")

        self.params = params

    @staticmethod
    def _clip(value: float, lower: float, upper: float) -> float:
        return max(lower, min(upper, value))

    def capacity_map(self, t_source_c: float, t_sink_c: float) -> float:
        """
        Compute the available maximum thermal power [kW].
        """
        p = self.params
        q_max_kw = (
            p.q_ref_kw
            + p.alpha_q_src * (t_source_c - p.t_source_ref_c)
            + p.alpha_q_sink * (t_sink_c - p.t_sink_ref_c)
        )
        return max(0.0, q_max_kw)

    def cop_map(self, t_source_c: float, t_sink_c: float) -> float:
        """
        Compute the heat pump COP [-].
        """
        p = self.params
        cop = (
            p.cop_ref
            + p.alpha_cop_src * (t_source_c - p.t_source_ref_c)
            + p.alpha_cop_sink * (t_sink_c - p.t_sink_ref_c)
        )
        return max(p.min_cop, cop)

    def evaluate(
        self,
        control_signal: float,
        t_source_c: float,
        t_sink_c: float,
    ) -> tuple[float, float, float, float]:
        """
        Evaluate the heat pump at the given operating point.

        Parameters
        ----------
        control_signal : float
            Normalized control input in [0, 1].
        t_source_c : float
            Source-side temperature [°C].
        t_sink_c : float
            Sink-side temperature [°C].

        Returns
        -------
        q_hp_kw : float
            Delivered thermal power [kW].
        p_hp_kw : float
            Electrical power consumption [kW].
        q_max_kw : float
            Maximum available thermal power [kW].
        cop : float
            Coefficient of performance [-].
        """
        p = self.params

        u = self._clip(control_signal, 0.0, 1.0)
        q_max_kw = self.capacity_map(t_source_c, t_sink_c)
        cop = self.cop_map(t_source_c, t_sink_c)

        if q_max_kw <= 0.0 or u <= 0.0:
            return 0.0, 0.0, q_max_kw, cop

        if 0.0 < u < p.min_modulation:
            u_effective = p.min_modulation
        else:
            u_effective = self._clip(u, p.min_modulation, p.max_modulation)

        q_hp_kw = u_effective * q_max_kw
        p_hp_kw = q_hp_kw / cop

        return q_hp_kw, p_hp_kw, q_max_kw, cop


# Minimal example
hp = QuasiSteadyHeatPump(
    QuasiSteadyHeatPumpParameters(
        q_ref_kw=10.0,
        cop_ref=3.5,
        t_source_ref_c=7.0,
        t_sink_ref_c=35.0,
        alpha_q_src=0.20,
        alpha_q_sink=-0.12,
        alpha_cop_src=0.08,
        alpha_cop_sink=-0.05,
        min_cop=1.1,
        min_modulation=0.30,
        max_modulation=1.0,
    )
)

u = 0.60
t_source_c = 5.0
t_sink_c = 40.0

q_hp_kw, p_hp_kw, q_max_kw, cop = hp.evaluate(
    control_signal=u,
    t_source_c=t_source_c,
    t_sink_c=t_sink_c,
)

print(f"Control signal: {u:.2f}")
print(f"Source temperature: {t_source_c:.1f} °C")
print(f"Sink temperature: {t_sink_c:.1f} °C")
print(f"Available thermal capacity: {q_max_kw:.2f} kW")
print(f"COP: {cop:.2f}")
print(f"Delivered thermal power: {q_hp_kw:.2f} kW")
print(f"Electrical power: {p_hp_kw:.2f} kW")


### Approach C — first-order dynamic extension

A natural future refinement is to include simple dynamics in the heat pump model. Rather than assuming that the delivered thermal power instantaneously follows the control signal, one may introduce a first-order lag or an equivalent dynamic state. This can help represent slower actuator response, smoother transitions, and more realistic interaction with a small storage volume.

At this stage, no implementation is required yet. The important point is simply to note that this extension exists and may become relevant if the control time step is short, if switching effects become important, or if a smoother representation of the HP-TES interaction is needed later in the project.
